[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# find and find_one &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's boot cell, with the `examined` helper. Run it first. Each task
opens its own client and closes it, so they can be run in any order.


In [1]:
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import pymongo

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")

def failed(error):
    """An OperationFailure's real message. Printing the exception whole would include a cluster
    time and a signature, which are different on every run and are never the point."""
    return f"{type(error).__name__}: {error.details.get('errmsg', error)}"


def examined(cursor):
    """What the server actually had to look at, which is the only honest measure of a query."""
    stats = cursor.explain()["executionStats"]
    return {"index keys": stats["totalKeysExamined"],
            "documents": stats["totalDocsExamined"],
            "returned": stats["nReturned"]}


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products")
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  500 products
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


**1.** One that is there and one that is not.


In [2]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

print("found:   ", shop.products.find_one({"_id": 12})["name"])
print("not found:", shop.products.find_one({"_id": 999999}))
client.close()


found:    Aster keyboard 12
not found: None


`None`, not an exception. Every `find_one` in a program needs to say what happens when the answer is
`None`, at the place it asks.


**2.** A cursor, twice.


In [3]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

cursor = shop.products.find({"kind": "mouse"})
print("first pass: ", len(list(cursor)))
print("second pass:", len(list(cursor)))
print("alive:", cursor.alive)
client.close()


first pass:  100
second pass: 0
alive: False


The cursor is exhausted, not broken. Nothing raised, which is why this bug usually surfaces as a
zero or an empty list somewhere far away.


**3.** Two fields and nothing else.


In [4]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

for product in shop.products.find({"kind": "cable"}, {"name": 1, "price": 1, "_id": 0}).limit(3):
    print(" ", product)
client.close()


  {'name': 'Aster cable 4', 'price': 1223.72}
  {'name': 'Belden cable 9', 'price': 584.21}
  {'name': 'Corvid cable 14', 'price': 205.71}


`"_id": 0` is the only exclusion allowed beside inclusions, and without it every document would
carry an `_id` you did not ask for.


**4.** The three most expensive.


In [5]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

for product in shop.products.find({}, {"name": 1, "price": 1, "_id": 0}) \
                            .sort("price", -1).limit(3):
    print(f"  {product['price']:8.2f}  {product['name']}")
client.close()


   1999.73  Corvid monitor 466
   1991.32  Dalgo cable 499
   1989.91  Corvid mouse 138


`sort` before `limit` in the chain, and in the server too: it sorts the whole matching set and then
takes three. Without an index on `price` that is a sort of every matching document, which
**Indexes** measures.


**5.** The same number, two ways.


In [6]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

print("count_documents:", shop.products.count_documents({"kind": "monitor"}))
print("estimated, whole collection:", shop.products.estimated_document_count())
print("the estimate cannot take a filter, which is the real difference")
client.close()


count_documents: 100
estimated, whole collection: 500
the estimate cannot take a filter, which is the real difference


`estimated_document_count` answers one question only: roughly how many documents are in this
collection. Anything with a filter has to be counted.


**6.** What skipping costs.


In [7]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

for skip in (0, 200):
    page = shop.products.find().sort("_id").skip(skip).limit(10)
    print(f"  skip={skip:3}", examined(page))
client.close()


  skip=  0 {'index keys': 10, 'documents': 10, 'returned': 10}
  skip=200 {'index keys': 210, 'documents': 10, 'returned': 10}


Ten index keys against two hundred and ten, for the same ten documents. The server walked past every
skipped entry, and it will walk past every one of them again on the next request for that page.


---

&#8592; **Back to:** [find and find_one](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/04-find-and-find-one.ipynb)  &nbsp;&middot;&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)
